# Build VneuroTK neural data

`BaseData` represents neural recordings while `NeuroData` is its neural-array container. `NeuroData` is not an `ndarray` subclass; use `.data` for the underlying array or `numpy.asarray()` for array conversion.

This notebook uses deterministic synthetic arrays and requires only the core package.

## Choose an explicit data mode

| Mode | Raw shape | Trial structure |
|---|---|---|
| `continuous` | `(n_samples, n_channels)` | Configure onsets before using `.epochs` |
| `epochs` | `(n_trials, n_timebins, n_channels)` | Already trial-structured |
| `patterns` | `(n_rows, n_channels)` | Aggregated rows |

Use the matching factory whenever a 2-D array's meaning would otherwise be ambiguous.

In [1]:
import numpy as np

import vneurotk as vtk

continuous = vtk.BaseData.for_continuous(
    neuro=np.arange(80, dtype=float).reshape(20, 4),
    neuro_info={"ch_names": ["a", "b", "c", "d"], "sfreq": 10.0},
)
epochs = vtk.BaseData.for_epochs(
    neuro=np.arange(96, dtype=float).reshape(3, 8, 4),
    neuro_info={"ch_names": ["a", "b", "c", "d"], "sfreq": 10.0},
)
patterns = vtk.BaseData.for_patterns(
    neuro=np.arange(24, dtype=float).reshape(6, 4),
    neuro_info={"ch_names": ["a", "b", "c", "d"]},
)
continuous.data_mode, epochs.data_mode, patterns.data_mode

('continuous', 'epochs', 'patterns')

## Inspect the neural container

In [2]:
neuro = patterns.neuro
neuro.shape, neuro.dtype, np.asarray(neuro).shape, neuro.data.shape

((6, 4), dtype('float64'), (6, 4), (6, 4))

## Configure continuous trial structure

In [3]:
stim_ids = np.array(["image-1", "image-2", "image-1"])
continuous.configure(
    vision_onsets=np.array([2, 8, 14]),
    stim_ids=stim_ids,
    vision_db={
        "image-1": np.zeros((8, 8, 3), dtype=np.uint8),
        "image-2": np.full((8, 8, 3), 255, dtype=np.uint8),
    },
    trial_window=[-1, 3],
)
continuous.neuro.epochs.shape

(3, 4, 4)

`configure()` binds stimulus IDs, onset samples, a trial window, and an image database. Pattern rows require `trial_meta["stim_index"]` only when they need alignment with vision features.

## Read recording sources

In [ ]:
from pathlib import Path

from vneurotk.io import EphysPath, MNEPath

# These examples describe inputs; reading requires the matching local files.
source_root = Path("data")
mne_source = MNEPath(
    root=source_root,
    subject="01",
    session="ImageNet01",
    task="ImageNet",
    run="01",
    suffix="meg_clean",
    extension=".fif",
)
ephys_source = EphysPath(
    root=source_root,
    session_id="251024_FanFan_nsd1w_MSB",
    dtype="TrialRaster",
    extension="h5",
)
# meg_data = vtk.read(mne_source)
# ephys_data = vtk.read(ephys_source)

Electrophysiology products are configured by their loaders. MNE recordings remain lazy until neural values are accessed and need `configure()` before trial-aligned views are available.

## Related documentation

- [Work with paths](path)
- [Extract vision features with neural data](vision_union)
- [Core API](../api/core.md)
- [Neural API](../api/neuro.md)